In [7]:
import torch
import numpy as np
from pydub import AudioSegment

In [8]:
# Load Silero VAD model
model = torch.jit.load('silero_vad.task')
_, utils = torch.hub.load(repo_or_dir='snakers4/silero-vad', model='silero_vad')
get_speech_timestamps, _, read_audio, _, _ = utils

Using cache found in C:\Users\beeko/.cache\torch\hub\snakers4_silero-vad_master


In [9]:

def analyze_speech_continuity(audio_path):
    # load audio
    audio = AudioSegment.from_file(audio_path).set_frame_rate(16000).set_channels(1).set_sample_width(2) # 16 bit
    # convert audio to tensor (format PyTorch 'Silero VAD' can work with)
    tensor = torch.tensor(np.array(audio.get_array_of_samples()).astype(np.float32) / 32767.0)

    # Detect speech segments
    speech_timestamps = get_speech_timestamps(tensor, model, sampling_rate=16000)

    # print(speech_timestamps) # uncomment to see the speech time list

    # Calculate percentage
    speech_duration = sum(ts['end'] - ts['start'] for ts in speech_timestamps)
    return round(speech_duration / len(tensor) * 100, 2)


In [10]:
result = analyze_speech_continuity("C:\\Users\\beeko\\PycharmProjects\\AI-Video-interview-analysis-\\DataSet\\audio.wav")
print(f"Speech continuity: {result}%")

Speech continuity: 68.21%


In [11]:
## this cell is basically about saving the model as Eng. Hossam asked us to :)
# # Saving the model with jit
# torch.jit.save(model, 'silero_vad.task')
#
# # Load it back
# model = torch.jit.load('silero_vad.task')
# model.eval()